# LSTM DQN Training

Uses a recurrent LSTM-based Q-network that maintains hidden state across timesteps.
Custom PyTorch training loop (SB3 DQN does not support recurrent networks natively).

In [ ]:
import os, sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir:     {DATA_DIR}')

In [ ]:
import numpy as np
import pandas as pd
from RLM.shared.data_loader import load_prices, load_trades

prices_df = load_prices(data_dir=DATA_DIR)
trades_df = load_trades(data_dir=DATA_DIR)

print(f'Prices: {len(prices_df)} rows')
print(f'Trades: {len(trades_df)} rows')
print(f'Products: {prices_df["product"].unique()}')
print(f'Days: {sorted(prices_df["day"].unique())}')

In [ ]:
from RLM.shared.config import PRODUCTS, TRAIN_CONFIG, DQN_CONFIG, NUM_FEATURES, NUM_ACTIONS
from RLM.shared.features import FeatureComputer, compute_features_from_row, fit_normalizer
from RLM.shared.env import TradingEnv

def compute_normalization_params(prices_df, trades_df, products, days):
    all_features = {p: [] for p in products}
    for day in days:
        for product in products:
            fc = FeatureComputer(product)
            day_prices = prices_df[(prices_df['day'] == day) & (prices_df['product'] == product)]
            day_prices = day_prices.sort_values('timestamp').reset_index(drop=True)
            day_trades = trades_df[trades_df['symbol'] == product].sort_values('timestamp')
            for _, row in day_prices.iterrows():
                ts = row['timestamp']
                ts_trades = day_trades[day_trades['timestamp'] == ts]
                trades = list(zip(ts_trades['price'], ts_trades['quantity'])) if len(ts_trades) > 0 else None
                features = compute_features_from_row(row, fc, position=0, trades=trades)
                all_features[product].append(features)
    combined = np.vstack([np.array(v) for v in all_features.values()])
    return fit_normalizer(combined)

feat_means, feat_stds = compute_normalization_params(
    prices_df, trades_df, PRODUCTS, TRAIN_CONFIG['train_days']
)
print(f'Normalization computed: {len(feat_means)} features')

## Train

Adjust `TOTAL_TIMESTEPS`, `HIDDEN_SIZE`, `SEQ_LEN` below.

In [ ]:
TOTAL_TIMESTEPS = 100_000
HIDDEN_SIZE = 64
SEQ_LEN = 10
SEED = 42
LEARNING_RATE = 1e-3

import torch
import torch.optim as optim
import random
from RLM.lstm_dqn.train import LSTMQNetwork, SequenceReplayBuffer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_env = TradingEnv(
    prices_df=prices_df, trades_df=trades_df,
    products=PRODUCTS, day=TRAIN_CONFIG['train_days'][0],
    augment=True, seed=SEED,
)
eval_env = TradingEnv(
    prices_df=prices_df, trades_df=trades_df,
    products=PRODUCTS, day=TRAIN_CONFIG['eval_days'][0],
    augment=False, seed=SEED + 1,
)
for product in PRODUCTS:
    train_env.feature_computers[product].feature_means = feat_means
    train_env.feature_computers[product].feature_stds = feat_stds
    eval_env.feature_computers[product].feature_means = feat_means
    eval_env.feature_computers[product].feature_stds = feat_stds

MODEL_DIR = os.path.join(PROJECT_ROOT, 'RLM', 'lstm_dqn', 'policy_weights')
os.makedirs(MODEL_DIR, exist_ok=True)

obs_dim = NUM_FEATURES * len(PRODUCTS)
action_dim = NUM_ACTIONS ** len(PRODUCTS)

q_net = LSTMQNetwork(obs_dim, action_dim, HIDDEN_SIZE).to(device)
target_net = LSTMQNetwork(obs_dim, action_dim, HIDDEN_SIZE).to(device)
target_net.load_state_dict(q_net.state_dict())
optimizer = optim.Adam(q_net.parameters(), lr=LEARNING_RATE)
replay_buffer = SequenceReplayBuffer(DQN_CONFIG['buffer_size'], seq_len=SEQ_LEN)

total_steps = 0
best_eval_pnl = -float('inf')
epsilon = DQN_CONFIG['exploration_initial_eps']
eps_decay = (DQN_CONFIG['exploration_initial_eps'] - DQN_CONFIG['exploration_final_eps']) / \
            (TOTAL_TIMESTEPS * DQN_CONFIG['exploration_fraction'])

print(f'Device: {device}, Hidden: {HIDDEN_SIZE}, Seq len: {SEQ_LEN}')
print(f'Training for {TOTAL_TIMESTEPS:,} timesteps...\n')

episode = 0
while total_steps < TOTAL_TIMESTEPS:
    obs, info = train_env.reset()
    hidden = q_net.init_hidden(1, device)
    done, ep_reward = False, 0

    while not done and total_steps < TOTAL_TIMESTEPS:
        if random.random() < epsilon:
            action = train_env.action_space.sample()
        else:
            with torch.no_grad():
                obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
                q_values, hidden = q_net(obs_t, hidden)
                action = q_values.argmax(dim=-1).item()

        next_obs, reward, terminated, truncated, info = train_env.step(action)
        done = terminated or truncated
        replay_buffer.add(obs, action, reward, next_obs, done)
        obs = next_obs
        ep_reward += reward
        total_steps += 1
        epsilon = max(DQN_CONFIG['exploration_final_eps'], epsilon - eps_decay)

        if len(replay_buffer) >= DQN_CONFIG['batch_size'] and total_steps % DQN_CONFIG['train_freq'] == 0:
            sequences = replay_buffer.sample(DQN_CONFIG['batch_size'])
            loss_total = 0
            for seq in sequences:
                states = torch.FloatTensor([s[0] for s in seq]).unsqueeze(0).to(device)
                actions = torch.LongTensor([s[1] for s in seq]).to(device)
                rewards = torch.FloatTensor([s[2] for s in seq]).to(device)
                next_states = torch.FloatTensor([s[3] for s in seq]).unsqueeze(0).to(device)
                dones = torch.FloatTensor([float(s[4]) for s in seq]).to(device)
                h = q_net.init_hidden(1, device)
                q_values, _ = q_net(states, h)
                q_value = q_values.squeeze(0)[actions[-1]]
                with torch.no_grad():
                    h_t = target_net.init_hidden(1, device)
                    next_q, _ = target_net(next_states, h_t)
                    target = rewards[-1] + DQN_CONFIG['gamma'] * next_q.squeeze(0).max() * (1 - dones[-1])
                loss_total += (q_value - target) ** 2
            loss = loss_total / len(sequences)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(q_net.parameters(), 10.0)
            optimizer.step()

        if total_steps % DQN_CONFIG['target_update_interval'] == 0:
            target_net.load_state_dict(q_net.state_dict())

    episode += 1
    if episode % 5 == 0:
        print(f'  Ep {episode}: steps={total_steps}, reward={ep_reward:.2f}, eps={epsilon:.3f}')

    if episode % 10 == 0:
        e_obs, _ = eval_env.reset()
        e_h = q_net.init_hidden(1, device)
        e_done, e_r = False, 0
        while not e_done:
            with torch.no_grad():
                e_q, e_h = q_net(torch.FloatTensor(e_obs).unsqueeze(0).to(device), e_h)
                e_a = e_q.argmax(dim=-1).item()
            e_obs, r, term, trunc, e_info = eval_env.step(e_a)
            e_r += r
            e_done = term or trunc
        e_pnl = e_info.get('pnl', e_r)
        print(f'  [EVAL] PnL={e_pnl:.2f}')
        if e_pnl > best_eval_pnl:
            best_eval_pnl = e_pnl
            torch.save(q_net.state_dict(), os.path.join(MODEL_DIR, 'best_model.pt'))
            print(f'  [EVAL] New best! Saved.')

torch.save(q_net.state_dict(), os.path.join(MODEL_DIR, 'final_model.pt'))
np.savez(os.path.join(MODEL_DIR, 'norm_params.npz'), feature_means=feat_means, feature_stds=feat_stds)
np.savez(os.path.join(MODEL_DIR, 'model_config.npz'),
         hidden_size=np.array(HIDDEN_SIZE), obs_dim=np.array(obs_dim), action_dim=np.array(action_dim))
print(f'\nTraining complete! Best eval PnL: {best_eval_pnl:.2f}')

## Evaluate

In [ ]:
best_path = os.path.join(MODEL_DIR, 'best_model.pt')
if os.path.exists(best_path):
    q_net.load_state_dict(torch.load(best_path, map_location=device))
    print('Loaded best model')

n_eval_episodes = 10
episode_pnls = []

for ep in range(n_eval_episodes):
    obs, info = eval_env.reset()
    hidden = q_net.init_hidden(1, device)
    total_reward, done = 0, False
    while not done:
        with torch.no_grad():
            obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
            q_values, hidden = q_net(obs_t, hidden)
            action = q_values.argmax(dim=-1).item()
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward
        done = terminated or truncated
    episode_pnls.append(info.get('pnl', total_reward))
    print(f'Episode {ep+1}: PnL={episode_pnls[-1]:.2f}')

pnls = np.array(episode_pnls)
print(f'\nMean PnL: {pnls.mean():.2f}  Std: {pnls.std():.2f}  Min: {pnls.min():.2f}  Max: {pnls.max():.2f}')
if pnls.std() > 0:
    print(f'Sharpe: {pnls.mean() / pnls.std():.2f}')

## Export Weights

Extracts LSTM gate weights and MLP head weights to `.npz`.

In [ ]:
weights = {'lstm_hidden_size': np.array(HIDDEN_SIZE)}

lstm = q_net.lstm
W_ih = lstm.weight_ih_l0.detach().cpu().numpy()
W_hh = lstm.weight_hh_l0.detach().cpu().numpy()
b_ih = lstm.bias_ih_l0.detach().cpu().numpy()
b_hh = lstm.bias_hh_l0.detach().cpu().numpy()
H = HIDDEN_SIZE

for idx, gate in enumerate(['i', 'f', 'g', 'o']):
    weights[f'lstm_W_i{gate}'] = W_ih[idx*H:(idx+1)*H]
    weights[f'lstm_W_h{gate}'] = W_hh[idx*H:(idx+1)*H]
    weights[f'lstm_b_{gate}'] = b_ih[idx*H:(idx+1)*H] + b_hh[idx*H:(idx+1)*H]
    print(f'  Gate {gate}: W_i {weights[f"lstm_W_i{gate}"].shape}, W_h {weights[f"lstm_W_h{gate}"].shape}')

head_idx = 0
for name, param in q_net.head.named_parameters():
    tensor = param.detach().cpu().numpy()
    print(f'  head.{name}: {tensor.shape}')
    if 'weight' in name: weights[f'head_W{head_idx}'] = tensor
    elif 'bias' in name: weights[f'head_B{head_idx}'] = tensor; head_idx += 1

weights['feature_means'] = feat_means
weights['feature_stds'] = feat_stds

export_path = os.path.join(MODEL_DIR, 'exported_weights.npz')
np.savez(export_path, **weights)
print(f'\nSaved to: {export_path} ({os.path.getsize(export_path) / 1024:.1f} KB)')

from RLM.shared.numpy_policy import NumpyLSTM
numpy_model = NumpyLSTM(weights_path=export_path)
action, q_vals = numpy_model.predict(np.random.randn(obs_dim).astype(np.float32), normalize=False)
print(f'Verification: action={action}, Q-values={q_vals[:3]}...')